# Étude des limites des avantages du PACS pour IR

### Paramétrage 

In [2]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from src.Ir_model import calculate_tax, calculate_tax_vectorized

# Plage de revenue net imposable annuel (salaire net imposable annuel pour faciliter la démarche) à considérer
min_rev = 0
max_rev = 200000

**Attention** : Les hypothèses prises pour réaliser ces estimations d'impôts sur le revenu sont relativement simplifiées. Les calculs d'estimation d'IR utilisés n'intègrent pas d'enfants, pas d'autres revenus (fonciers, capitaux), pas de réductions/crédits d'impôt, abattement forfaitaire des frais professionnels (salarié) pas au réel (TNS) et le revenu supposé est le "salaire net imposable annuel".

L'objectif de cet exercice est de démontrer de manière pédagogique les limites des avantages fiscaux pouvant être apportés par le PACS concernant l'IR. 

Je cherche à démontrer que **le PACS/mariage n'est pas systématiquement avantageux fiscalement**, et non une simulation exhaustive du calcul d'impôts sur le revenu.

*Je ne suis en aucun cas fiscaliste et je n'ai pas la prétention de me revendiquer comme telle.*

### Exemple d'avantage sans prendre en compte l'écart de revenu entre partenaires

In [ ]:
x = np.linspace(min_rev, max_rev*1.5, 101)

tax_concubinage = []
tax_pacsé = []

for i in x:
    tax_concubinage.append(calculate_tax(i/2)+calculate_tax(i/2))
    tax_pacsé.append(calculate_tax(i, tax_individual=False, quotient_familial=2))
    
fig, (axe1,axe2) = plt.subplots(1, 2, figsize=(18, 6))
axe1.plot(x, tax_concubinage, label='Concubinage',color = 'black',linestyle='dashed')
axe1.plot(x, tax_pacsé, label='PACS - Mariage', color = 'orange')
axe1.set_title("Revenu équilibré entre deux déclarants")
axe1.set_xlabel("Revenu total")
axe1.set_ylabel("Impôt sur le revenu")
axe1.legend()

tax_concubinage = []
tax_pacsé = []

for i in x:
    tax_concubinage.append(calculate_tax(i*0.2)+calculate_tax(i*0.8))
    tax_pacsé.append(calculate_tax(i, tax_individual=False, quotient_familial=2))

axe2.plot(x, tax_concubinage, label='Concubinage',color = 'black',linestyle='dashed')
axe2.plot(x, tax_pacsé, label='PACS - Mariage', color = 'orange')
axe2.set_title("Revenu déséquilibré entre deux déclarants (20/80)")
axe2.set_xlabel("Revenu total")
axe2.set_ylabel("Impôt sur le revenu")
axe2.legend()

plt.show()
    

En effet les graphiques ci-dessus démontrent qu'il y a un fort avantage fiscal à privilégier le PACS ou le mariage pour le paiement de l'impôt sur le revenu, lorsque les dits revenus des deux déclarants sont différents. 

Dans notre exemple (graphique à droite) l'un des revenus est 5 fois supérieur à l'autre, ce qui représente un écart significatif et nous permet d'identifier un vrai avantage fiscal entre le statut de concubin et celui de pacsé ou marié.

La question que nous pouvons nous poser à présent est, **quelles sont les limites à cet avantage fiscal ?** 

Pouvons-nous mesurer, anticiper et représenter graphiquement ces limites sous forme d'équation.

Et si oui comment le faire ?

## Définition de la surface de l'impôt sur le revenu concubin

In [ ]:
def plot_surface_concu(min_rev, max_rev):
    x = np.linspace(min_rev, max_rev, 101)
    y = np.linspace(min_rev, max_rev, 101)

    X, Y = np.meshgrid(x, y)

    Z1 = calculate_tax_vectorized(X)
    Z2 = calculate_tax_vectorized(Y)
    Z = Z1 + Z2

    fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')])
    fig.update_layout(title='Surface de l\'impôt sur le revenu concubinage', scene=dict(
        xaxis_title='Revenu 1',yaxis_title='Revenu 2',zaxis_title='Impôt total'),width=900, height=700)
    return fig, Z

fig, tax_concubinage = plot_surface_concu(min_rev, max_rev)
fig.show()

## Définition de la surface de l'impôt sur le revenu PACS

In [ ]:
def plot_surface_pascé(min_rev, max_rev):
    x = np.linspace(min_rev, max_rev, 101)
    y = np.linspace(min_rev, max_rev, 101)

    X, Y = np.meshgrid(x, y)

    Z = calculate_tax_vectorized((X + Y), tax_individual=False, quotient_familial=2) 
    fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y,colorscale='Viridis')])
    fig.update_layout(title='Surface de l\'impôt sur le revenu pacsé', scene=dict(
        xaxis_title='Revenu 1',yaxis_title='Revenu 2',zaxis_title='Impôt total'),width=900, height=700)
    return fig, Z

fig, tax_pacsé = plot_surface_pascé(min_rev, max_rev)
fig.show()



## Représentation graphique des limites des avantages fiscaux du PACS

En réalisant une soustraction entre ces deux surfaces, nous pouvons constater la surface de l'écart d'impôts sur le revenu entre les deux statuts. On remarque ainsi que comparé au graphique de comparaison précédemment présenté, les zones où l'on retrouve un avantage fiscal ne sont pas systématiques.

In [ ]:
def plot_surface_compar(min_rev, max_rev,tax_concubinage,tax_pacsé):
    x = np.linspace(min_rev, max_rev, 101)
    y = np.linspace(min_rev, max_rev, 101)
    X, Y = np.meshgrid(x, y)

    Z = tax_concubinage - tax_pacsé

    fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y,colorscale='Viridis')])
    fig.update_layout(title='Gain fiscal pacsé vs concubinage', scene=dict(
        xaxis_title='Revenu 1',yaxis_title='Revenu 2',zaxis_title='Gain sur IR'),width=900, height=700)
    return fig, Z

fig, tax_pacsé = plot_surface_compar(min_rev, max_rev,tax_concubinage,tax_pacsé)
fig.show()

On observe ici une représentation graphique 3D des seuils d'optimisation fiscale pouvant être apportés par le PACS ou le mariage. En effet, on constate des surfaces planes en forme de carré qui délimitent clairement des zones où il n'y a aucun avantage fiscal d'être pacsé (voir dans certaines zones un désavantage).

Oui, le PACS dans certaines conditions peut représenter une véritable source d'optimisation fiscale face à l'IR, le fait est que cet avantage est obtenable que sous certaines conditions d'écart de revenu, représenté avec le graphique 3D ci-dessus.

Cependant, il est important de constater que dans certain cas (la deuxième surface plane carrée au niveau des revenus les plus faibles), on remarque que le PACS est désavantageux face au concubinage, généralement pour un montant de 311€. Cela est induit par le mécanisme de décote de l'IR qui est plus avantageux pour un couple concubin qu'un couple pacsé. Pour des concubins, la décote forfaitaire individuelle est multipliée par deux, soit 2 x 897€ = 1 794€, alors que pour un couple pacsé, elle est appliquée sous forme de forfait couple de 1 483€. Cet écart présent de 311€ est donc uniquement causé par cet écart de calcul de décote entre les deux situations.

## Simplifions l'analyse

Vous trouverez ci-dessous une représentation graphique 2D (heatmap, plus lisible) de l'avantage fiscal obtenable grâce au PACS en fonction du revenu des deux déclarants. Avec également une simulation de votre situation (étoile bleue).

In [13]:
# Revenu pour exemple
revenu_1 = 30000
revenu_2 = 20000

In [ ]:
impot_concubin_exemple = calculate_tax(revenu_1) + calculate_tax(revenu_2)
impot_pacsé_exemple = calculate_tax(revenu_1 + revenu_2, tax_individual=False, quotient_familial=2)
ecart_exemple = impot_concubin_exemple - impot_pacsé_exemple

def plot_heatmap_gain_pacs(min_rev, max_rev, step=10000):

    # Grille de revenus
    x = np.arange(min_rev, max_rev + step, step)
    y = np.arange(min_rev, max_rev + step, step)
    
    X, Y = np.meshgrid(x, y)
    
    # Calcul impôts
    impot_concubin = calculate_tax_vectorized(X) + calculate_tax_vectorized(Y)
    impot_pacsé = calculate_tax_vectorized(X + Y, tax_individual=False, quotient_familial=2)
    
    # Gain possible en PACS par rapport au concubinage
    gain = impot_concubin - impot_pacsé
    
    # Créer la figure
    fig = go.Figure()
    
    # Heatmap avec Contour (pour graphique dynamique)
    fig.add_trace(go.Contour(
        z=gain,
        x=X[0],
        y=Y[:, 0],
        colorscale='RdYlGn',
        colorbar=dict(title="Gain fiscal (€)",
                       tickprefix="€",
                       tickfont=dict(size=10)),
        showscale=True,
        contours=dict(
            showlabels=True,
            labelfont=dict(size=10, color='white')
        ),
        hovertemplate="Revenu 1: %{x:.0f}€<br>Revenu 2: %{y:.0f}€<br>Gain PACS: %{z:.0f}€<extra></extra>"
    ))
    
    # AJout des ligne au niveau 0 pour séparer les zones de gain et de perte
    fig.add_trace(go.Contour(
        z=gain,
        x=X[0],
        y=Y[:, 0],
        colorscale='RdYlGn',
        showscale=False,
        # levels=[0],  # Uniquement le niveau 0
        line=dict(color='black', width=4, dash='solid'),
        contours=dict(showlabels=True),
        name='Seuil gain/perte (0€)',
        hoverinfo='skip'
    ))
    
    # Ajouter du point d'exemple
    fig.add_trace(go.Scatter(
        x=[revenu_1],
        y=[revenu_2],
        mode='markers+text',
        marker=dict(size=15, color='blue', symbol='star', 
                   line=dict(color='darkblue', width=2)),
        text=[f'Votre exemple<br> {revenu_1/1000}k/{revenu_2/1000}k <br>Gain: {ecart_exemple:.0f}€'],
        textposition='top center',
        textfont=dict(size=11, color='blue'),
        name='Votre situation',
        hovertemplate=f'Votre cas: {revenu_1/1000}k/{revenu_2/1000}k <br>Gain: {ecart_exemple:.0f}€<br>'
    ))
    
    # Configurer la mise en page
    fig.update_layout(
        title='Gain fiscal : PACS vs Concubinage (Revenus 2025)',
        xaxis_title='Revenu Partenaire 1 (€)',
        yaxis_title='Revenu Partenaire 2 (€)',
        width=950,
        height=850,
        hovermode='closest',
        plot_bgcolor='white',
        xaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray'),
        yaxis=dict(showgrid=True, gridwidth=1, gridcolor='lightgray')
    )
    
    return fig, gain

# Lancer la visualisation
fig, gain_matrix = plot_heatmap_gain_pacs(0, 200000, step=2000)
fig.show()

# Statistiques
print("\n=== ANALYSE DE LA HEATMAP ===")
gain_max = np.nanmax(gain_matrix)
gain_min = np.nanmin(gain_matrix)
print(f"Gain max (meilleur cas PACS): {gain_max:.0f}€")
print(f"Perte max (pire cas PACS): {gain_min:.0f}€")

zone_pacs_favorable = np.sum(gain_matrix > 0)
zone_pacs_defavorable = np.sum(gain_matrix < 0)
total = gain_matrix.size

print(f"\nPoints où PACS est favorable (gain): {zone_pacs_favorable / total * 100:.1f}%")
print(f"Points où PACS est défavorable (perte): {zone_pacs_defavorable / total * 100:.1f}%")

# Exemple : trouver le gain à un revenu spécifique
print(f"\nPour exemple choisi pour le revenu 1 {revenu_1/1000}k€ et le revenu 2 {revenu_2/1000}k€ = {ecart_exemple:.0f}€")

In [ ]:
concubinage = calculate_tax(revenu_1) + calculate_tax(revenu_2)
pacse = calculate_tax(revenu_1 + revenu_2, tax_individual=False, quotient_familial=2)

print(f"Impôt concubinage : {concubinage:.0f} €")
print(f"Impôt PACS : {pacse:.0f} €")
print(f"Gain du PACS : {concubinage - pacse:.0f} €")